In [6]:
1 - Basis Script uitvoeren

In [ ]:
#collect_sysinfo.py ; output in json format
import json
import psutil
import platform
import socket

import datetime
now = datetime.datetime.now()
print(now)

def collect_system_info():
    data = {}

    # CPU info
    data["cpu"] = {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_processors": psutil.cpu_count(logical=True),
        "cpu_freq": psutil.cpu_freq()._asdict() if psutil.cpu_freq() else None,
        "cpu_percent_total": psutil.cpu_percent(interval=0.1),
        "cpu_percent_per_cpu": psutil.cpu_percent(interval=0.1, percpu=True),
    }

    # Memory info
    data["memory"] = psutil.virtual_memory()._asdict()
    data["swap"] = psutil.swap_memory()._asdict()

    # Storage info (alle partities)
    storage = {}
    for part in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(part.mountpoint)
            storage[part.mountpoint] = {
                "device": part.device,
                "fstype": part.fstype,
                "opts": part.opts,
                "total": usage.total,
                "used": usage.used,
                "free": usage.free,
                "percent": usage.percent,
            }
        except PermissionError:
            continue
    data["storage"] = storage

    # Network info
    net = {}
    addrs = psutil.net_if_addrs()
    stats = psutil.net_if_stats()
    io = psutil.net_io_counters(pernic=True)
    for iface, addr_list in addrs.items():
        net[iface] = {
            "addrs": [a._asdict() for a in addr_list],
            "stats": stats[iface]._asdict() if iface in stats else None,
            "io": io[iface]._asdict() if iface in io else None,
        }

    data["network"] = {
        "hostname": socket.gethostname(),
        "fqdn": socket.getfqdn(),
        "interfaces": net,
    }

    return data


if __name__ == "__main__":
    info = collect_system_info()
    # JSON output
    print(json.dumps(info, indent=2))

2026-10-07 09:42:15.353675
{
  "cpu": {
    "platform": "Windows-11-10.0.26200-SP0",
    "processor": "Intel64 Family 6 Model 186 Stepping 2, GenuineIntel",
    "physical_cores": 14,
    "logical_processors": 20,
    "cpu_freq": {
      "current": 2400.0,
      "min": 0.0,
      "max": 2400.0
    },
    "cpu_percent_total": 25.9,
    "cpu_percent_per_cpu": [
      16.7,
      0.0,
      0.0,
      0.0,
      33.3,
      0.0,
      16.7,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0
    ]
  },
  "memory": {
    "total": 16858472448,
    "available": 4378947584,
    "percent": 74.0,
    "used": 12479524864,
    "free": 4378947584
  },
  "swap": {
    "total": 42949672960,
    "used": 1000226816,
    "free": 41949446144,
    "percent": 2.3,
    "sin": 0,
    "sout": 0
  },
  "storage": {
    "C:\\": {
      "device": "C:\\",
      "fstype": "NTFS",
      "opts": "rw,fixed",
      "total": 10227

In [ ]:
2 - Informatie filteren

In [8]:
import json
import os
import platform
import socket
import subprocess

import psutil


GIB = 1024 ** 3


def computermodel():
    """Zit niet in de basisdata, dus apart opvragen per besturingssysteem."""
    try:
        if platform.system() == "Windows":
            import winreg
            with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                                r"HARDWARE\DESCRIPTION\System\BIOS") as k:
                return (winreg.QueryValueEx(k, "SystemManufacturer")[0] + " " +
                        winreg.QueryValueEx(k, "SystemProductName")[0])
        if platform.system() == "Linux":
            with open("/sys/class/dmi/id/sys_vendor") as f1, \
                 open("/sys/class/dmi/id/product_name") as f2:
                return f1.read().strip() + " " + f2.read().strip()
        if platform.system() == "Darwin":
            return subprocess.check_output(["sysctl", "-n", "hw.model"], text=True).strip()
    except OSError:
        return None


def primair_ip():
    """IPv4-adres dat het OS gebruikt om internet te bereiken (default route).
    Een UDP-connect verstuurt geen pakketten."""
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.connect(("8.8.8.8", 80))
        return s.getsockname()[0]


def filter_info(data):
    # Opslag: de partitie met het besturingssysteem
    if platform.system() == "Windows":
        schijf = os.environ.get("SystemDrive", "C:") + "\\"
    else:
        schijf = "/"
    opslag = data["storage"].get(schijf, {})

    # Primaire interface: de interface die het uitgaande IPv4-adres heeft
    ip = primair_ip()
    mac = ipv6 = ipv6_ll = None
    for naam, info in data["network"]["interfaces"].items():
        adressen = [a["address"] for a in info["addrs"] if a["family"] == socket.AF_INET]
        if ip not in adressen:
            continue
        for a in info["addrs"]:
            adres = a["address"].split("%")[0]  # zone-index (%eth0) weghalen
            if a["family"] == psutil.AF_LINK:
                mac = adres.replace("-", ":").lower()
            elif a["family"] == socket.AF_INET6:
                if adres.startswith("fe80"):
                    ipv6_ll = adres
                else:
                    ipv6 = adres
        break

    freq = data["cpu"]["cpu_freq"]
    return {
        "computernaam": data["network"]["hostname"],
        "computermodel": computermodel(),
        "processor": data["cpu"]["processor"],
        "cpu_fysieke_cores": data["cpu"]["physical_cores"],
        "cpu_max_frequentie_mhz": freq["max"] if freq else None,
        "ram_totaal_gib": round(data["memory"]["total"] / GIB, 2),
        "opslag_bestandssysteem": opslag.get("fstype"),
        "opslag_capaciteit_gib": round(opslag["total"] / GIB, 2) if opslag else None,
        "mac_adres": mac,
        "ipv4_adres": ip,
        "ipv6_adres": ipv6,
        "ipv6_link_local": ipv6_ll,
    }


if __name__ == "__main__":
    print(json.dumps(filter_info(collect_system_info()), indent=2))

{
  "computernaam": "milan",
  "computermodel": "HP Victus by HP Gaming Laptop 16-r0xxx",
  "processor": "Intel64 Family 6 Model 186 Stepping 2, GenuineIntel",
  "cpu_fysieke_cores": 14,
  "cpu_max_frequentie_mhz": 2400.0,
  "ram_totaal_gib": 15.7,
  "opslag_bestandssysteem": "NTFS",
  "opslag_capaciteit_gib": 952.55,
  "mac_adres": "10:68:38:40:a8:c3",
  "ipv4_adres": "10.110.177.20",
  "ipv6_adres": null,
  "ipv6_link_local": "fe80::b359:907d:aa86:91dd"
}


In [ ]:
3- JSON

In [ ]:
Zet je gefilterde dictionary om naar JSON.
Bewaar de gegevens in een .json-bestand.
Lees het bestand opnieuw in als een Python dictionary.
Controleer na het opnieuw inlezen of de belangrijkste gegevens overeenkomen met de oorspronkelijke dictionary.
Leg in enkele zinnen concreet uit wat het verschil is tussen een Python dict en JSON.

In [9]:
import json

gefilterd = filter_info(collect_system_info())

# 1. dict -> JSON-tekst
json_tekst = json.dumps(gefilterd, indent=2)

# 2. JSON-tekst opslaan in een bestand
with open("sysinfo.json", "w", encoding="utf-8") as f:
    f.write(json_tekst)

# 3. Bestand opnieuw inlezen als dict
with open("sysinfo.json", encoding="utf-8") as f:
    ingelezen = json.load(f)

# 4. Controle per sleutel en in zijn geheel
for sleutel, waarde in gefilterd.items():
    status = "OK" if ingelezen.get(sleutel) == waarde else "VERSCHIL"
    print(f"{status:8} {sleutel}: {ingelezen.get(sleutel)}")
print("Volledig gelijk:", gefilterd == ingelezen)

OK       computernaam: milan
OK       computermodel: HP Victus by HP Gaming Laptop 16-r0xxx
OK       processor: Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
OK       cpu_fysieke_cores: 14
OK       cpu_max_frequentie_mhz: 2400.0
OK       ram_totaal_gib: 15.7
OK       opslag_bestandssysteem: NTFS
OK       opslag_capaciteit_gib: 952.55
OK       mac_adres: 10:68:38:40:a8:c3
OK       ipv4_adres: 10.110.177.20
OK       ipv6_adres: None
OK       ipv6_link_local: fe80::b359:907d:aa86:91dd
Volledig gelijk: True


In [ ]:
4 - Transformeren van informatie naar CSV
De gefilterde informatie bevindt zich momenteel in een Python dict. In deze opdracht transformeer je deze gegevens naar een CSV-bestand (Comma-Separated Values) dat in een spreadsheetprogramma kan worden geopend.

Bepaal een geschikte bestandsnaam.
Bepaal welke informatie je in de kolommen wilt opnemen en kies duidelijke kolomnamen.
Bepaal hoe de informatie over één computer als één rij in het CSV-bestand kan worden voorgesteld.
Transformeer de informatie uit de Python dictionary naar een tabelstructuur die geschikt is voor CSV.
Schrijf de gegevens met Python naar een .csv-bestand.
Open het CSV-bestand in een spreadsheetprogramma en controleer of de gegevens correct worden weergegeven.
Zorg ervoor dat je script het CSV-bestand later opnieuw kan inlezen.
Denk hierbij na over:

Welk scheidingsteken gebruik je in het CSV-bestand?
Hoe ga je om met ontbrekende informatie?
Hoe ga je om met informatie waarvan een computer meerdere exemplaren kan hebben, zoals disks of netwerkinterfaces?
Hoe zorg je ervoor dat het CSV-bestand correct kan worden geopend in je spreadsheetprogramma?

In [11]:
import csv
import os
 
CSV_BESTAND = "sysinfo_computers.csv"
SCHEIDINGSTEKEN = ";" 
 
KOLOMMEN = [
    "computernaam", "computermodel", "processor", "cpu_fysieke_cores",
    "cpu_max_frequentie_mhz", "ram_totaal_gib", "opslag_bestandssysteem",
    "opslag_capaciteit_gib", "mac_adres", "ipv4_adres", "ipv6_adres",
    "ipv6_link_local",
]
# Kolommen die bij het inlezen terug een getal moeten worden (CSV bevat enkel tekst)
GETALLEN = {
    "cpu_fysieke_cores": int,
    "cpu_max_frequentie_mhz": float,
    "ram_totaal_gib": float,
    "opslag_capaciteit_gib": float,
}
 
 
def naar_cel(waarde):
    """Python-waarde -> tekst in de CSV."""
    if waarde is None:
        return ""    
    if isinstance(waarde, float):
        return str(waarde).replace(".", ",") # decimale komma Excel
    return str(waarde)
 
def van_cel(kolom, tekst):
    """Tekst uit de CSV -> Python-waarde."""
    if tekst == "":
        return None
    if kolom in GETALLEN:
        return GETALLEN[kolom](tekst.replace(",", "."))
    return tekst
 
 
def lees_csv(bestand=CSV_BESTAND):
    """Leest het CSV-bestand in als een lijst van dicts (één dict per computer)."""
    if not os.path.exists(bestand):
        return []
    with open(bestand, newline="", encoding="utf-8-sig") as f:
        lezer = csv.DictReader(f, delimiter=SCHEIDINGSTEKEN)
        return [{k: van_cel(k, v) for k, v in rij.items()} for rij in lezer]
 
 
def schrijf_csv(rijen, bestand=CSV_BESTAND):
    """Schrijft een lijst van dicts weg: één rij per computer."""
    with open(bestand, "w", newline="", encoding="utf-8-sig") as f:
        schrijver = csv.DictWriter(f, fieldnames=KOLOMMEN, delimiter=SCHEIDINGSTEKEN)
        schrijver.writeheader()
        for rij in rijen:
            schrijver.writerow({k: naar_cel(rij.get(k)) for k in KOLOMMEN})
 
 
# Deze computer toevoegen, staat hij er al in, dan wordt zijn rij vervangen.
gefilterd = filter_info(collect_system_info())
rijen = [r for r in lees_csv() if r["computernaam"] != gefilterd["computernaam"]]
rijen.append(gefilterd)
schrijf_csv(rijen)
 
# Controle: opnieuw inlezen en vergelijken met de oorspronkelijke dict
mijn_rij = next(r for r in lees_csv() if r["computernaam"] == gefilterd["computernaam"])
print("Rij gelijk aan oorspronkelijke dict:", mijn_rij == gefilterd)

Rij gelijk aan oorspronkelijke dict: True


In [ ]:
5 - Transformeren naar een spreadsheetbestand
Zet de gegevens uit taak 4 met Python om naar een spreadsheetbestand.

Kies een geschikt spreadsheetformaat.
Zoek uit welke Python-module of library je hiervoor kunt gebruiken.
Gebruik dezelfde gegevens en kolomnamen als in het CSV-bestand.
Bewaar het resultaat in een spreadsheetbestand.
Open het bestand in een spreadsheetprogramma en controleer het resultaat.
Vergelijk deze werkwijze met het gebruik van CSV.
Denk hierbij na over:

Wat is het verschil tussen een CSV-bestand en een echt spreadsheetbestand?
Welke extra mogelijkheden biedt een spreadsheetformaat?
Wanneer zou je CSV verkiezen en wanneer een spreadsheetbestand?

In [12]:
# Plak in een notebookcel NA de cel van taak 4 (gebruikt KOLOMMEN en lees_csv).
# Eenmalig installeren indien nodig:  %pip install openpyxl
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

XLSX_BESTAND = "sysinfo_computers.xlsx"


def schrijf_xlsx(rijen, bestand=XLSX_BESTAND):
    wb = Workbook()
    ws = wb.active
    ws.title = "Computers"

    # Gegevens: zelfde kolommen als de CSV, één rij per computer.
    # None wordt een lege cel; getallen blijven echte getallen.
    ws.append(KOLOMMEN)
    for rij in rijen:
        ws.append([rij.get(k) for k in KOLOMMEN])

    # Opmaak: dit kan CSV niet
    for rij in ws.iter_rows():
        for cel in rij:
            cel.font = Font(name="Arial")
    for cel in ws[1]:  # kopregel
        cel.font = Font(name="Arial", bold=True, color="FFFFFF")
        cel.fill = PatternFill("solid", fgColor="305496")
    for kolom in ("ram_totaal_gib", "opslag_capaciteit_gib"):
        letter = get_column_letter(KOLOMMEN.index(kolom) + 1)
        for cel in ws[letter][1:]:
            cel.number_format = "0.00"  # Excel toont zelf , of . volgens de taalinstelling
    for i in range(1, len(KOLOMMEN) + 1):
        letter = get_column_letter(i)
        breedte = max(len(str(c.value or "")) for c in ws[letter])
        ws.column_dimensions[letter].width = breedte + 2
    ws.freeze_panes = "A2"               # kopregel blijft zichtbaar bij scrollen
    ws.auto_filter.ref = ws.dimensions   # filterknoppen op de kopregel

    wb.save(bestand)


def lees_xlsx(bestand=XLSX_BESTAND):
    """Leest het werkblad in als een lijst van dicts, net zoals lees_csv()."""
    ws = load_workbook(bestand).active
    rijen = list(ws.iter_rows(values_only=True))
    kop = rijen[0]
    return [dict(zip(kop, rij)) for rij in rijen[1:]]


# CSV inlezen (taak 4) -> spreadsheetbestand schrijven -> controleren
rijen = lees_csv()
schrijf_xlsx(rijen)
print("Gegevens in XLSX gelijk aan CSV:", lees_xlsx() == rijen)

Gegevens in XLSX gelijk aan CSV: True


In [ ]:
6

In [ ]:
Ieder groepslid heeft nu een gestructureerd bestand met de informatie over zijn of haar eigen computer. Maak met Python één databestand waarin de informatie van de computers van alle groepsleden wordt samengevoegd.

Bepaal samen welke kolommen in het gemeenschappelijke CSV-bestand worden gebruikt.
Zorg ervoor dat alle groepsleden hun gegevens volgens dezelfde structuur opslaan.
Lees met Python de afzonderlijke CSV-bestanden van de groepsleden in.
Voeg de gegevens automatisch samen tot één dataset.
Schrijf het resultaat naar een nieuw CSV-bestand.
Het is niet toegestaan om de gegevens manueel te kopiëren.
Open het uiteindelijke CSV-bestand in een spreadsheetprogramma en controleer het resultaat.
Denk hierbij na over:

Hoe zorg je ervoor dat de kolomnamen bij iedereen hetzelfde zijn?
Wat gebeurt er wanneer gegevens bij een groepslid ontbreken?
Hoe kan je script bepalen welke CSV-bestanden moeten worden ingelezen?
Hoe zorg je ervoor dat nieuwe computers later eenvoudig aan de dataset kunnen worden toegevoegd?
Spreek binnen het team af hoe en waar de afzonderlijke bestanden worden uitgewisseld en hoe de bestanden worden benoemd.

In [13]:
# Voeg de CSV-bestanden van alle groepsleden samen tot één dataset.
# Uitvoeren in je notebook met:  %run -i samenvoegen.py
# (NA de cel van taak 4: gebruikt KOLOMMEN, SCHEIDINGSTEKEN, lees_csv en schrijf_csv)
import csv
import glob
import os

INVOERMAP = "groepsdata"                                 # gedeelde map met alle bestanden
PATROON = os.path.join(INVOERMAP, "sysinfo_*.csv")       # afgesproken bestandsnaam
UITVOER = "sysinfo_groep.csv"                            # staat BUITEN de invoermap


def lees_kopregel(bestand):
    with open(bestand, newline="", encoding="utf-8-sig") as f:
        return next(csv.reader(f, delimiter=SCHEIDINGSTEKEN), [])


samengevoegd = {}  # computernaam -> rij; zo komt elke computer maar één keer voor
bestanden = sorted(glob.glob(PATROON))
print(f"{len(bestanden)} bestanden gevonden in '{INVOERMAP}'")

for bestand in bestanden:
    kop = lees_kopregel(bestand)
    ontbrekend = [k for k in KOLOMMEN if k not in kop]
    extra = [k for k in kop if k not in KOLOMMEN]

    if "computernaam" not in kop:
        print(f"  OVERGESLAGEN {bestand}: kolommen niet herkend (verkeerd scheidingsteken?)")
        continue
    if ontbrekend:
        print(f"  LET OP {bestand}: kolommen ontbreken {ontbrekend} -> lege cellen")
    if extra:
        print(f"  LET OP {bestand}: onbekende kolommen {extra} -> genegeerd")

    for rij in lees_csv(bestand):
        naam = rij.get("computernaam")
        if not naam:
            print(f"  OVERGESLAGEN rij zonder computernaam in {bestand}")
            continue
        if naam in samengevoegd:
            print(f"  LET OP: '{naam}' staat in meerdere bestanden, versie uit {bestand} gebruikt")
        samengevoegd[naam] = rij

schrijf_csv(list(samengevoegd.values()), UITVOER)

# Controle: resultaat opnieuw inlezen en ontbrekende gegevens tonen
resultaat = lees_csv(UITVOER)
print(f"\n{len(resultaat)} computers samengevoegd in '{UITVOER}'")
for kolom in KOLOMMEN:
    leeg = [r["computernaam"] for r in resultaat if r.get(kolom) is None]
    if leeg:
        print(f"  {kolom} ontbreekt bij: {', '.join(leeg)}")

0 bestanden gevonden in 'groepsdata'

0 computers samengevoegd in 'sysinfo_groep.csv'


In [ ]:
7

In [14]:
import ipaddress
import re
from collections import Counter
 
DATASET = "sysinfo_groep.csv"
MAC_PATROON = re.compile(r"^([0-9a-f]{2}:){5}[0-9a-f]{2}$")
 
 
def controleer_rij(r):
    """Geeft een lijst met problemen terug voor één computer."""
    fouten = []
    # Formaat
    if r["mac_adres"] and not MAC_PATROON.match(r["mac_adres"]):
        fouten.append(f"ongeldig MAC-adres {r['mac_adres']}")
    try:
        if r["ipv4_adres"] and ipaddress.IPv4Address(r["ipv4_adres"]).is_link_local:
            fouten.append("IPv4 is 169.254.x.x (geen DHCP-adres gekregen)")
    except ValueError:
        fouten.append(f"ongeldig IPv4-adres {r['ipv4_adres']}")
    if r["ipv6_link_local"] and not r["ipv6_link_local"].startswith("fe80"):
        fouten.append("link-local adres begint niet met fe80")
    # Plausibiliteit (vangt ook verkeerde eenheden, bv. bytes in plaats van GiB)
    if r["ram_totaal_gib"] is not None and not 1 <= r["ram_totaal_gib"] <= 512:
        fouten.append(f"onwaarschijnlijke RAM: {r['ram_totaal_gib']} GiB")
    if r["opslag_capaciteit_gib"] is not None and not 16 <= r["opslag_capaciteit_gib"] <= 20000:
        fouten.append(f"onwaarschijnlijke opslag: {r['opslag_capaciteit_gib']} GiB")
    if r["cpu_fysieke_cores"] is not None and not 1 <= r["cpu_fysieke_cores"] <= 128:
        fouten.append(f"onwaarschijnlijk aantal cores: {r['cpu_fysieke_cores']}")
    return fouten
 
 
dataset = lees_csv(DATASET)
 
# 1. Controles per computer
for r in dataset:
    for fout in controleer_rij(r):
        print(f"  {r['computernaam']}: {fout}")
 
# 2. Controles over de hele dataset: dingen die uniek moeten zijn
for kolom in ("computernaam", "mac_adres"):
    telling = Counter(r[kolom] for r in dataset if r[kolom])
    for waarde, aantal in telling.items():
        if aantal > 1:
            print(f"  {kolom} '{waarde}' komt {aantal} keer voor")
 
# 3. Eigen computer: opgeslagen gegevens vergelijken met een nieuwe meting
nu = filter_info(collect_system_info())
opgeslagen = next((r for r in dataset if r["computernaam"] == nu["computernaam"]), None)
if opgeslagen is None:
    print(f"  Deze computer ({nu['computernaam']}) staat niet in de dataset")
else:
    for kolom, waarde in nu.items():
        if opgeslagen.get(kolom) != waarde:
            print(f"  Verschil bij {kolom}: dataset={opgeslagen.get(kolom)}  nu={waarde}")
print("Controle klaar.")

  Deze computer (milan) staat niet in de dataset
Controle klaar.
